# 🧠 EEG Stress Detection — Enhanced CBGG (TensorFlow/Keras)

**Base Paper:** Roy et al. (2023) — CBGG Hybrid Architecture  
**Dataset:** STEW — loaded from Kaggle input (`/kaggle/input/...`)  
**Task:** 3-class → `Low (0)` | `Medium (1)` | `High (2)`  
**Split:** Subject-dependent segment-level 70/30 (matches paper)

---
## Pipeline
```
dataset.mat  +  class_012.mat
      ↓
Expand labels: 45 subjects → 19200 segments
      ↓
DWT Decomposition (db4, max safe level)
→ bands: approx / alpha / beta / gamma
      ↓
Feature Extraction
  [CBGG]  Mean, Variance, Skewness, Kurtosis, Power
  [OURS]  Differential Entropy, Hjorth, Katz FD, Band Ratios
      ↓
Min-Max Normalization
      ↓
CNN-1D (filters=128, kernel=1, activation=softmax)
      ↓
BiLSTM (units=64)
      ↓
GRU    (units=32)
      ↓
GRU    (units=16)
      ↓
Dropout (0.2)
      ↓
Dense → Low / Medium / High
```

## Our Novelty
Extended CBGG features + ablation study proving each feature group's contribution.

## 📦 0. Install Dependencies

In [1]:
# Run once then comment out
!pip install PyWavelets scikit-learn scipy matplotlib seaborn

## ⚙️ 1. Configuration

In [ ]:
import numpy as np
import random
import os

# -- TensorFlow runtime logging controls ------------------
# Must be set before importing tensorflow.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_CPP_MIN_LOG_LEVEL"] = "3"

# -- Dataset ---------------------------------------------
SFREQ       = 128
N_CHANNELS  = 14
N_CLASSES   = 3
CLASS_NAMES = ["Low", "Medium", "High"]

# -- Kaggle paths ----------------------------------------
# Keep outputs in /kaggle/working so they are saved in notebook output.
DATA_DIR        = "/kaggle/input"
CHECKPOINT_DIR  = "/kaggle/working/checkpoints"
RESULTS_DIR     = "/kaggle/working/results"

# -- DWT -------------------------------------------------
WAVELET    = "db4"
# DWT_LEVEL set automatically in Cell 5 based on signal length

# -- CBGG Architecture (exact from paper) ----------------
CNN_FILTERS  = 128
CNN_KERNEL   = 1
BILSTM_UNITS = 64
GRU1_UNITS   = 32
GRU2_UNITS   = 16
DROPOUT      = 0.2

# -- Training (Kaggle-tuned) -----------------------------
BASE_BATCH_SIZE       = 32
BATCH_SIZE            = 64  # auto-updated after GPU strategy setup
EPOCHS                = 100
LEARNING_RATE         = 1e-3
EFFECTIVE_LR          = LEARNING_RATE
EARLY_STOP_PATIENCE   = 12
EARLY_STOP_MIN_DELTA  = 1e-4
N_FOLDS               = 10
RANDOM_SEED           = 42

# -- Feature flags ---------------------------------------
USE_CBGG_FEATURES = True
USE_OUR_FEATURES  = True

print(f"Classes    : {N_CLASSES}  ({' | '.join(CLASS_NAMES)})")
print(f"Channels   : {N_CHANNELS}")
print(f"Wavelet    : {WAVELET}")
print(f"Epochs     : {EPOCHS}")
print(f"Base batch : {BASE_BATCH_SIZE}")

Classes  : 3  (Low | Medium | High)
Channels : 14
Wavelet  : db4


## 🔁 2. Reproducibility

In [ ]:
# Safety: ensure log level is set before TensorFlow init in this kernel.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("ABSL_CPP_MIN_LOG_LEVEL", "3")

import tensorflow as tf
from tensorflow.keras import mixed_precision

# Optional speedups for Kaggle GPU runtime.
tf.config.optimizer.set_jit(True)

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed()

# Discover and configure GPUs.
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

if len(gpus) > 1:
    STRATEGY = tf.distribute.MirroredStrategy()
else:
    STRATEGY = tf.distribute.get_strategy()

NUM_REPLICAS = STRATEGY.num_replicas_in_sync

# Mixed precision accelerates Tensor Core GPUs such as T4.
if gpus:
    mixed_precision.set_global_policy("mixed_float16")

# Scale batch size by the number of replicas.
BATCH_SIZE = BASE_BATCH_SIZE * NUM_REPLICAS
EFFECTIVE_LR = LEARNING_RATE * max(1, NUM_REPLICAS)

print(f"TensorFlow      : {tf.__version__}")
print(f"GPUs detected   : {len(gpus)}")
print(f"Replicas in sync: {NUM_REPLICAS}")
print(f"Mixed precision : {mixed_precision.global_policy().name}")
print(f"XLA JIT         : enabled")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Learning rate   : {EFFECTIVE_LR}")
print(f"Seed {RANDOM_SEED} fixed")

TensorFlow : 2.19.0
Seed 42 fixed ✓


## 📂 3. Kaggle Paths & Dataset Discovery

In [ ]:
from pathlib import Path

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,    exist_ok=True)

print("Kaggle working dirs ready ✓")
print(f"CHECKPOINT_DIR: {CHECKPOINT_DIR}")
print(f"RESULTS_DIR   : {RESULTS_DIR}")
print("\nAvailable Kaggle input datasets:")
for p in Path("/kaggle/input").glob("*"):
    print(" -", p)

Mounted at /content/drive
Drive mounted ✓


In [ ]:
import scipy.io
from pathlib import Path

def find_stew_files(root="/kaggle/input"):
    root = Path(root)
    dataset_candidates = list(root.rglob("dataset.mat"))
    label_candidates   = list(root.rglob("class_012.mat"))

    if not dataset_candidates or not label_candidates:
        raise FileNotFoundError(
            "Could not find dataset.mat and class_012.mat under /kaggle/input"
        )

    # Prefer same directory pair when possible.
    for d in dataset_candidates:
        for l in label_candidates:
            if d.parent == l.parent:
                return d, l

    return dataset_candidates[0], label_candidates[0]


def load_stew():
    eeg_file, lbl_file = find_stew_files(DATA_DIR)
    print(f"dataset.mat path   : {eeg_file}")
    print(f"class_012.mat path : {lbl_file}")

    eeg_mat = scipy.io.loadmat(str(eeg_file))
    lbl_mat = scipy.io.loadmat(str(lbl_file))

    print("dataset.mat   keys:", [k for k in eeg_mat if not k.startswith("_")])
    print("class_012.mat keys:", [k for k in lbl_mat if not k.startswith("_")])

    eeg_key = [k for k in eeg_mat if not k.startswith("_")][0]
    lbl_key = [k for k in lbl_mat if not k.startswith("_")][0]

    X = eeg_mat[eeg_key]
    y = lbl_mat[lbl_key].flatten().astype(np.int64)

    print(f"\nRaw X shape  : {X.shape}")
    print(f"Raw y shape  : {y.shape}")
    print(f"Unique labels: {np.unique(y)}")
    print(f"Label counts : {np.bincount(y)}  (Low / Medium / High)")
    return X, y

X_raw, y_subjects = load_stew()

dataset.mat   keys: ['dataset']
class_012.mat keys: ['class_012']

Raw X shape  : (14, 19200, 45)
Raw y shape  : (45,)
Unique labels: [0 1 2]
Label counts : [13 18 14]  (Low / Medium / High)


## 🔧 4. Fix Shape & Expand Labels

In [5]:
# ── Fix EEG shape to (n_segments, n_channels, n_samples) ─────
X = X_raw.copy().astype(np.float32)

if X.ndim == 3:
    if X.shape[1] != N_CHANNELS and X.shape[2] == N_CHANNELS:
        X = X.transpose(0, 2, 1)
        print("Transposed → (segments, channels, samples)")
    elif X.shape[0] == N_CHANNELS:
        X = X.transpose(1, 0, 2)
        print("Transposed → (segments, channels, samples)")

n_segments, n_ch, n_samp = X.shape
print(f"X shape          : {X.shape}")
print(f"Segments         : {n_segments}")
print(f"Channels         : {n_ch}")
print(f"Samples/segment  : {n_samp}  ({n_samp/SFREQ:.3f} sec)")

# ── Expand labels: 45 subjects → 19200 segments ──────────────
n_subjects  = len(y_subjects)    # 45
base        = n_segments // n_subjects
extra       = n_segments %  n_subjects

counts = np.array([
    base + 1 if i < extra else base
    for i in range(n_subjects)
])
assert counts.sum() == n_segments, "Count mismatch!"

y = np.repeat(y_subjects, counts).astype(np.int64)

print(f"\ny shape          : {y.shape}")
print(f"Label dist       : {np.bincount(y)}  (Low / Medium / High)")
print(f"  Low    (0)     : {(y==0).sum()}")
print(f"  Medium (1)     : {(y==1).sum()}")
print(f"  High   (2)     : {(y==2).sum()}")

Transposed → (segments, channels, samples)
X shape          : (19200, 14, 45)
Segments         : 19200
Channels         : 14
Samples/segment  : 45  (0.352 sec)

y shape          : (19200,)
Label dist       : [5547 7682 5971]  (Low / Medium / High)
  Low    (0)     : 5547
  Medium (1)     : 7682
  High   (2)     : 5971


## 🌊 5. DWT Decomposition
Decompose each channel using DWT (db4). Level set automatically based on signal length.

In [6]:
import pywt

# ── Auto-set safe DWT level ───────────────────────────────────
max_level = pywt.dwt_max_level(X.shape[-1], WAVELET)
DWT_LEVEL = min(max_level, 3)
print(f"Signal length  : {X.shape[-1]} samples")
print(f"Max DWT level  : {max_level}")
print(f"Using level    : {DWT_LEVEL}")

# Band names based on actual level
if DWT_LEVEL == 4:
    BAND_NAMES = ["delta", "theta", "alpha", "beta", "gamma"]
elif DWT_LEVEL == 3:
    BAND_NAMES = ["approx", "alpha", "beta", "gamma"]
elif DWT_LEVEL == 2:
    BAND_NAMES = ["approx", "beta", "gamma"]
else:
    BAND_NAMES = [f"band_{i}" for i in range(DWT_LEVEL + 1)]

print(f"Band names     : {BAND_NAMES}")


def decompose_dataset_fast(X):
    """
    Vectorized DWT — processes all segments per channel at once.
    14 iterations instead of 19200.
    Returns: {band: array (n_segments, n_channels, coeff_len)}
    """
    band_data = {b: [] for b in BAND_NAMES}

    for ch in range(X.shape[1]):
        ch_signals  = X[:, ch, :]   # (n_segments, n_samples)
        coeffs_list = pywt.wavedec(ch_signals, wavelet=WAVELET,
                                   level=DWT_LEVEL, axis=-1)
        for b_idx, band in enumerate(BAND_NAMES):
            band_data[band].append(coeffs_list[b_idx])

    for band in BAND_NAMES:
        band_data[band] = np.stack(band_data[band], axis=1).astype(np.float32)
        print(f"  {band:8s} shape: {band_data[band].shape}")

    return band_data


print("\nRunning DWT decomposition (vectorized)...")
band_data = decompose_dataset_fast(X)
print("DWT complete ✓")

Signal length  : 45 samples
Max DWT level  : 2
Using level    : 2
Band names     : ['approx', 'beta', 'gamma']

Running DWT decomposition (vectorized)...
  approx   shape: (19200, 14, 16)
  beta     shape: (19200, 14, 16)
  gamma    shape: (19200, 14, 26)
DWT complete ✓


## 🔬 6. Feature Extraction
### CBGG Features (paper)
Mean, Variance, Skewness, Kurtosis, Power — per band per channel

### Our Features (novelty)
Differential Entropy, Hjorth Parameters, Katz FD, Band Power Ratios

In [7]:
from scipy import stats as sp_stats

def extract_cbgg_features_fast(band_data):
    """
    Vectorized CBGG features:
    mean, variance, skewness, kurtosis, power — per band per channel
    """
    all_bands = []
    for band in BAND_NAMES:
        data  = band_data[band]              # (N, C, coeff_len)
        mean  = data.mean(axis=-1)
        var   = data.var(axis=-1)
        power = (data ** 2).mean(axis=-1)
        skew  = sp_stats.skew(data, axis=-1)
        kurt  = sp_stats.kurtosis(data, axis=-1)
        band_feats = np.stack([mean, var, skew, kurt, power], axis=-1)
        all_bands.append(band_feats.reshape(len(data), -1))

    feats = np.concatenate(all_bands, axis=1).astype(np.float32)
    print(f"CBGG features    : {feats.shape}  "
          f"({len(BAND_NAMES)} bands x {N_CHANNELS} ch x 5)")
    return feats


def extract_our_features_fast(band_data):
    """
    Vectorized our features:
    Differential Entropy, Hjorth (act/mob/cmp), Katz FD — per band per channel
    Band power ratios — per channel
    """
    all_parts = []

    for band in BAND_NAMES:
        data = band_data[band]   # (N, C, coeff_len)

        # Differential Entropy
        var = data.var(axis=-1) + 1e-12
        de  = 0.5 * np.log(2 * np.pi * np.e * var)

        # Hjorth parameters
        dx  = np.diff(data, axis=-1)
        ddx = np.diff(dx,   axis=-1)
        act = data.var(axis=-1)
        mob = dx.std(axis=-1)   / (data.std(axis=-1) + 1e-12)
        cmp = (ddx.std(axis=-1) / (dx.std(axis=-1)   + 1e-12)) / (mob + 1e-12)

        # Katz Fractal Dimension
        L   = np.abs(np.diff(data, axis=-1)).sum(axis=-1)
        d   = np.abs(data - data[:, :, [0]]).max(axis=-1)
        n   = data.shape[-1]
        kfd = np.log10(n) / (np.log10(n) + np.log10(d / (L + 1e-12)))

        band_feats = np.stack([de, act, mob, cmp, kfd], axis=-1)
        all_parts.append(band_feats.reshape(len(data), -1))

    # Band power ratios (highest / lowest and mid / lowest)
    band_powers = {
        band: (band_data[band] ** 2).mean(axis=-1)
        for band in BAND_NAMES
    }
    highest = band_powers[BAND_NAMES[-1]]
    lowest  = band_powers[BAND_NAMES[0]]
    ratio1  = (highest / (lowest + 1e-12)).reshape(len(highest), -1)
    all_parts.append(ratio1)

    if len(BAND_NAMES) >= 3:
        mid    = band_powers[BAND_NAMES[len(BAND_NAMES)//2]]
        ratio2 = (mid / (lowest + 1e-12)).reshape(len(mid), -1)
        all_parts.append(ratio2)

    feats = np.concatenate(all_parts, axis=1).astype(np.float32)
    print(f"Our features     : {feats.shape}")
    return feats


print("Extracting CBGG features...")
feats_cbgg = extract_cbgg_features_fast(band_data)

print("Extracting our features...")
feats_ours = extract_our_features_fast(band_data)

if USE_CBGG_FEATURES and USE_OUR_FEATURES:
    features = np.concatenate([feats_cbgg, feats_ours], axis=1)
    mode_str = "CBGG + Our Features (Enhanced)"
elif USE_CBGG_FEATURES:
    features = feats_cbgg
    mode_str = "CBGG Features Only (Baseline)"
else:
    features = feats_ours
    mode_str = "Our Features Only"

print(f"\nMode           : {mode_str}")
print(f"Feature matrix : {features.shape}")

Extracting CBGG features...
CBGG features    : (19200, 210)  (3 bands x 14 ch x 5)
Extracting our features...
Our features     : (19200, 238)

Mode           : CBGG + Our Features (Enhanced)
Feature matrix : (19200, 448)


## 📏 7. Min-Max Normalization

In [8]:
from sklearn.preprocessing import MinMaxScaler

# Global scaler for inspection only
# Per-fold scaling done inside CV to prevent leakage
scaler_global = MinMaxScaler()
features_norm = scaler_global.fit_transform(features).astype(np.float32)

print(f"Feature range before : [{features.min():.3f}, {features.max():.3f}]")
print(f"Feature range after  : [{features_norm.min():.3f}, {features_norm.max():.3f}]")
print(f"Shape                : {features_norm.shape}")

Feature range before : [-294.023, 89412208.000]
Feature range after  : [0.000, 1.000]
Shape                : (19200, 448)


## 🏗️ 8. CBGG Model (Keras)
Exact architecture from paper Table 2:
- Conv1D(128, kernel=1, activation=softmax)
- MaxPooling1D(pool_size=1)
- BiLSTM(64)
- GRU(32)
- GRU(16)
- Dropout(0.2)
- Dense(n_classes, softmax)

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, Bidirectional,
    LSTM, GRU, Dropout, Dense, Activation
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)
import tensorflow.keras.backend as K


def build_cbgg(input_size, n_classes=N_CLASSES):
    """
    CBGG model — exact paper architecture.
    Input: (input_size, 1)
    """
    inputs = Input(shape=(input_size, 1), name="input")

    # CNN-1D (paper: filter=128, kernel=1, activation=softmax)
    x = Conv1D(filters=CNN_FILTERS, kernel_size=CNN_KERNEL,
               padding="valid", name="conv1d")(inputs)
    x = Activation("softmax", name="softmax_act")(x)
    x = MaxPooling1D(pool_size=1, name="maxpool")(x)

    # BiLSTM (units=64)
    x = Bidirectional(
        LSTM(units=BILSTM_UNITS, return_sequences=True),
        name="bilstm"
    )(x)

    # GRU 1 (units=32)
    x = GRU(units=GRU1_UNITS, return_sequences=True,  name="gru1")(x)

    # GRU 2 (units=16)
    x = GRU(units=GRU2_UNITS, return_sequences=False, name="gru2")(x)

    # Dropout (0.2)
    x = Dropout(DROPOUT, name="dropout")(x)

    # Keep output in float32 for numeric stability with mixed precision.
    outputs = Dense(n_classes, activation="softmax", dtype="float32", name="output")(x)

    return Model(inputs=inputs, outputs=outputs, name="CBGG")


# -- Test build -------------------------------------------
n_features = features.shape[1]
with STRATEGY.scope():
    test_model = build_cbgg(n_features)
test_model.summary()

print(f"\nPaper params : 117,657")
print(f"Our params   : {test_model.count_params():,}")

Model: "CBGG"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 448, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 448, 128)       │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax_act (Activation)        │ (None, 448, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool (MaxPooling1D)          │ (None, 448, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm (Bidirectional)          │ (None, 448, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru1 (GRU)                      │ (None, 448, 32)        │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru2 (GRU)                      │ (None, 16)             │         2,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 117,075 (457.32 KB)

 Trainable params: 117,075 (457.32 KB)

 Non-trainable params: 0 (0.00 B)


Paper params : 117,657
Our params   : 117,075


## 🔢 9. DataLoader Utilities

In [10]:
from tensorflow.keras.utils import to_categorical

def prepare_fold(X_tr, X_vl, y_tr, y_vl):
    """
    Scale + reshape + one-hot encode for one fold.
    Scaler fit on train only — no leakage.
    """
    sc   = MinMaxScaler()
    X_tr = sc.fit_transform(X_tr).astype(np.float32)
    X_vl = sc.transform(X_vl).astype(np.float32)

    # Reshape for Conv1D: (samples, features, 1)
    X_tr_3d = X_tr[:, :, np.newaxis]
    X_vl_3d = X_vl[:, :, np.newaxis]

    y_tr_cat = to_categorical(y_tr, N_CLASSES)
    y_vl_cat = to_categorical(y_vl, N_CLASSES)

    counts   = np.bincount(y_tr)
    class_wts = {i: len(y_tr) / (N_CLASSES * c)
                 for i, c in enumerate(counts)}

    return X_tr_3d, X_vl_3d, y_tr_cat, y_vl_cat, class_wts


print("Utilities defined ✓")

Utilities defined ✓


## 🏋️ 10. Training & Evaluation Utilities

In [11]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)


def get_callbacks(ckpt_path):
    return [
        ModelCheckpoint(ckpt_path, monitor="val_loss",
                        save_best_only=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                          patience=7, min_lr=1e-6, verbose=0),
    ]


def eval_fold(model, X_vl_3d, y_vl):
    probs = model.predict(X_vl_3d, verbose=0)
    preds = probs.argmax(axis=1)
    acc   = accuracy_score(y_vl, preds)
    f1    = f1_score(y_vl, preds, average="weighted", zero_division=0)
    try:
        auc_val = roc_auc_score(y_vl, probs,
                                 multi_class="ovr", average="macro")
    except ValueError:
        auc_val = 0.0
    return probs, preds, acc, f1, auc_val


print("Training utilities defined ✓")

Training utilities defined ✓


## 🔄 11. 10-Fold Cross Validation
**Subject-dependent segment-level split — matches paper exactly.**  
Stratified 10-fold on all 19200 segments.

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
print("Physical GPUs:", gpus)
print("Logical GPUs :", tf.config.list_logical_devices('GPU'))
print("Replicas     :", NUM_REPLICAS)
if len(gpus) >= 2:
    print("Dual-GPU MirroredStrategy active ✓")
else:
    print("Single GPU runtime detected; training still works.")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
/device:GPU:0


In [ ]:
from sklearn.model_selection import StratifiedKFold

set_seed()

# -- Segment-level stratified 10-fold (matches paper split) --
skf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                            random_state=RANDOM_SEED)

cv_results = []
all_preds  = np.zeros(len(y), dtype=np.int64)
all_probs  = np.zeros((len(y), N_CLASSES), dtype=np.float32)

for fold, (train_idx, val_idx) in enumerate(skf.split(features, y)):

    print(f"\n{'='*55}")
    print(f"  Fold {fold+1}/{N_FOLDS}")
    print(f"  Train segments: {len(train_idx)}  Val segments: {len(val_idx)}")
    print(f"{'='*55}")

    X_tr_3d, X_vl_3d, y_tr_cat, y_vl_cat, class_wts = prepare_fold(
        features[train_idx], features[val_idx],
        y[train_idx],        y[val_idx]
    )

    # Build fresh model under strategy scope.
    K.clear_session()
    set_seed()

    with STRATEGY.scope():
        fold_model = build_cbgg(n_features)
        fold_model.compile(
            optimizer=Adam(learning_rate=EFFECTIVE_LR),
            loss="categorical_crossentropy",
            metrics=["accuracy"],
        )

    # Smart stopping: allow LR scheduler to react first, then stop on plateau.
    callbacks = [
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),
        EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOP_PATIENCE,
            min_delta=EARLY_STOP_MIN_DELTA,
            mode="min",
            restore_best_weights=True,
            verbose=1
        ),
    ]

    history = fold_model.fit(
        X_tr_3d, y_tr_cat,
        validation_data=(X_vl_3d, y_vl_cat),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_wts,
        callbacks=callbacks,
        verbose=1,
    )

    # Evaluate best epoch model (restored by EarlyStopping)
    probs, preds, acc, f1, auc_val = eval_fold(
        fold_model, X_vl_3d, y[val_idx]
    )

    all_preds[val_idx] = preds
    all_probs[val_idx] = probs

    cv_results.append({
        "fold"      : fold + 1,
        "accuracy"  : float(acc),
        "f1"        : float(f1),
        "auc"       : float(auc_val),
        "train_acc" : history.history["accuracy"],
        "val_acc"   : history.history["val_accuracy"],
        "train_loss": history.history["loss"],
        "val_loss"  : history.history["val_loss"],
    })

    # Save after every fold
    with open(f"{RESULTS_DIR}/cv_results_partial.json", "w") as f:
        json.dump(cv_results, f, indent=2,
                  default=lambda x: float(x)
                  if isinstance(x, np.floating) else x)

    epochs_ran = len(history.history["accuracy"])

    print(f"  Epochs ran : {epochs_ran}")
    print(f"  Accuracy   : {acc*100:.2f}%")
    print(f"  F1 (wtd)   : {f1:.4f}")
    print(f"  AUC (OvR)  : {auc_val:.4f}")
    print(f"  Saved ✓")


# -- CV Summary -------------------------------------------
accs = [r["accuracy"] for r in cv_results]
f1s  = [r["f1"]       for r in cv_results]
aucs = [r["auc"]      for r in cv_results]

print(f"\n{'='*55}")
print(f"  10-FOLD CV RESULTS (subject-dependent segment split)")
print(f"{'='*55}")
print(f"  Accuracy : {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%")
print(f"  F1 (wtd) : {np.mean(f1s):.4f}  ± {np.std(f1s):.4f}")
print(f"  AUC (OvR): {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"  Paper CV : 97.60%")

with open(f"{RESULTS_DIR}/cv_results.json", "w") as f:
    json.dump(cv_results, f, indent=2,
              default=lambda x: float(x)
              if isinstance(x, np.floating) else x)

print("CV results saved ✓")


  Fold 1/10
  Train segments: 17280  Val segments: 1920
Epoch 1/200
346/346 ━━━━━━━━━━━━━━━━━━━━ 30s 65ms/step - accuracy: 0.3357 - loss: 1.0995 - val_accuracy: 0.4000 - val_loss: 1.0935 - learning_rate: 0.0010
Epoch 2/200
346/346 ━━━━━━━━━━━━━━━━━━━━ 21s 61ms/step - accuracy: 0.3239 - loss: 1.0992 - val_accuracy: 0.4000 - val_loss: 1.0956 - learning_rate: 0.0010
Epoch 3/200
346/346 ━━━━━━━━━━━━━━━━━━━━ 22s 62ms/step - accuracy: 0.3243 - loss: 1.0991 - val_accuracy: 0.4000 - val_loss: 1.0946 - learning_rate: 0.0010
Epoch 4/200
346/346 ━━━━━━━━━━━━━━━━━━━━ 21s 59ms/step - accuracy: 0.3503 - loss: 1.0986 - val_accuracy: 0.4141 - val_loss: 1.0821 - learning_rate: 0.0010
Epoch 5/200
346/346 ━━━━━━━━━━━━━━━━━━━━ 21s 62ms/step - accuracy: 0.3692 - loss: 1.0870 - val_accuracy: 0.4234 - val_loss: 1.0762 - learning_rate: 0.0010
Epoch 6/200
346/346 ━━━━━━━━━━━━━━━━━━━━ 20s 59ms/step - accuracy: 0.3706 - loss: 1.0848 - val_accuracy: 0.4302 - val_loss: 1.0746 - learning_rate: 0.0010
Epoch 7/200
3

## 📊 12. Training Convergence Curves

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes      = axes.flatten()

for i, r in enumerate(cv_results):
    ep = range(1, len(r["train_acc"]) + 1)
    axes[i].plot(ep, r["train_acc"], label="Train", color="steelblue",  lw=1.5)
    axes[i].plot(ep, r["val_acc"],   label="Val",   color="tomato",     lw=1.5)
    axes[i].set_title(f"Fold {r['fold']}  (acc={r['accuracy']*100:.1f}%)")
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel("Accuracy")
    axes[i].legend(fontsize=7)
    axes[i].set_ylim(0, 1.05)
    axes[i].grid(alpha=0.3)

plt.suptitle(
    f"Train vs Validation Accuracy — 10-Fold CV\n"
    f"Mean: {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%",
    fontsize=13
)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/convergence_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Convergence curves saved ✓")

## 🎯 13. Overall Confusion Matrix

In [ ]:
print("Classification Report (all folds combined):")
print(classification_report(y, all_preds, target_names=CLASS_NAMES))

cm  = confusion_matrix(y, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True",      fontsize=12)
ax.set_title("Confusion Matrix — 10-Fold CV", fontsize=13)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/confusion_matrix.png", dpi=150)
plt.show()
print("Confusion matrix saved ✓")

## 📈 14. ROC Curve

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_bin  = label_binarize(y, classes=[0, 1, 2])
colors = ["steelblue", "darkorange", "mediumseagreen"]

fig, ax = plt.subplots(figsize=(8, 6))
for i, (cls, color) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f"{cls} (AUC = {roc_auc:.3f})")

ax.plot([0,1],[0,1],"k--", lw=1, label="Random")
ax.set_xlim([0,1]); ax.set_ylim([0,1.05])
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate",  fontsize=12)
ax.set_title("ROC Curve — One vs Rest (10-Fold CV)", fontsize=13)
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/roc_curve.png", dpi=150)
plt.show()
print("ROC curve saved ✓")

## 📋 15. Full Metrics Table
All metrics from paper Table 3: Precision, Sensitivity, Specificity, F1, Accuracy, +LR, -LR

In [ ]:
import pandas as pd

def compute_paper_metrics(y_true, y_pred, class_names):
    cm   = confusion_matrix(y_true, y_pred)
    rows = []
    for i, cls in enumerate(class_names):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - TP - FP - FN

        precision   = TP / (TP + FP + 1e-12) * 100
        sensitivity = TP / (TP + FN + 1e-12) * 100
        specificity = TN / (TN + FP + 1e-12) * 100
        f1_val      = 2*precision*sensitivity / (precision+sensitivity+1e-12)
        acc_val     = (TP + TN) / cm.sum() * 100
        plr         = sensitivity / (100 - specificity + 1e-12)
        nlr         = (100 - sensitivity) / (specificity + 1e-12)

        rows.append({
            "Class"      : cls,
            "Precision"  : f"{precision:.2f}%",
            "Sensitivity": f"{sensitivity:.2f}%",
            "Specificity": f"{specificity:.2f}%",
            "F1-Score"   : f"{f1_val:.2f}%",
            "Accuracy"   : f"{acc_val:.2f}%",
            "+LR"        : f"{plr:.3f}",
            "-LR"        : f"{nlr:.3f}",
        })

    overall = accuracy_score(y_true, y_pred) * 100
    return pd.DataFrame(rows), overall


metrics_df, overall_acc = compute_paper_metrics(y, all_preds, CLASS_NAMES)
print(f"Overall Accuracy : {overall_acc:.2f}%")
print(f"Paper Accuracy   : 98.10%")
print(f"\nPer-Class Metrics:")
print(metrics_df.to_string(index=False))

metrics_df.to_csv(f"{RESULTS_DIR}/metrics_table.csv", index=False)
print("\nMetrics table saved ✓")

## 🔬 16. Ablation Study
Prove our features add value over original CBGG features.  
5-fold CV for speed.

In [ ]:
ablation_configs = [
    ("CBGG Only (Baseline)",   feats_cbgg),
    ("Our Features Only",      feats_ours),
    ("CBGG + Ours (Enhanced)", np.concatenate([feats_cbgg, feats_ours], axis=1)),
]

ablation_results = {}
skf_ab = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

for mode_name, feat_set in ablation_configs:
    print(f"\n{'─'*50}")
    print(f"  {mode_name}  ({feat_set.shape[1]} features)")
    print(f"{'─'*50}")

    fold_accs = []
    for fold, (train_idx, val_idx) in enumerate(
            skf_ab.split(feat_set, y)):

        X_tr_3d, X_vl_3d, y_tr_cat, y_vl_cat, class_wts = prepare_fold(
            feat_set[train_idx], feat_set[val_idx],
            y[train_idx],        y[val_idx]
        )

        K.clear_session()
        set_seed()
        with STRATEGY.scope():
            ab_model = build_cbgg(feat_set.shape[1])
            ab_model.compile(
                optimizer=Adam(EFFECTIVE_LR),
                loss="categorical_crossentropy",
                metrics=["accuracy"]
            )
        ab_model.fit(
            X_tr_3d, y_tr_cat,
            validation_data=(X_vl_3d, y_vl_cat),
            epochs=35, batch_size=BATCH_SIZE,
            class_weight=class_wts,
            callbacks=[EarlyStopping(patience=8,
                                     restore_best_weights=True)],
            verbose=0
        )

        _, _, acc, _, _ = eval_fold(ab_model, X_vl_3d, y[val_idx])
        fold_accs.append(acc)
        print(f"  Fold {fold+1}: {acc*100:.2f}%")

    mean_acc = np.mean(fold_accs)
    std_acc  = np.std(fold_accs)
    ablation_results[mode_name] = {
        "mean_acc"   : float(mean_acc),
        "std_acc"    : float(std_acc),
        "n_features" : feat_set.shape[1],
    }
    print(f"  Mean: {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")


print(f"\n{'='*50}")
print("  ABLATION SUMMARY")
print(f"{'='*50}")
for name, res in ablation_results.items():
    print(f"  {name:35s} | "
          f"features={res['n_features']:4d} | "
          f"acc={res['mean_acc']*100:.2f}% ± {res['std_acc']*100:.2f}%")

with open(f"{RESULTS_DIR}/ablation_results.json", "w") as f:
    json.dump(ablation_results, f, indent=2)
print("Ablation results saved ✓")

## 📊 17. Ablation Chart

In [ ]:
names  = list(ablation_results.keys())
means  = [ablation_results[n]["mean_acc"] * 100 for n in names]
stds   = [ablation_results[n]["std_acc"]  * 100 for n in names]
n_feat = [ablation_results[n]["n_features"]     for n in names]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, means, yerr=stds, capsize=6,
              color=["steelblue", "darkorange", "mediumseagreen"],
              alpha=0.85, edgecolor="black", width=0.5)

for bar, mean, n in zip(bars, means, n_feat):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            f"{mean:.2f}%\n({n} feats)",
            ha="center", va="bottom", fontsize=9)

ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_ylim(0, 120)
ax.set_title("Ablation Study — Feature Contribution", fontsize=13)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/ablation_chart.png", dpi=150)
plt.show()
print("Ablation chart saved ✓")

## 💾 18. Final Summary

In [ ]:
summary = {
    "model"          : "Enhanced CBGG (CNN-1D + BiLSTM + GRU + GRU)",
    "dataset"        : "STEW",
    "n_classes"      : N_CLASSES,
    "class_names"    : CLASS_NAMES,
    "split_method"   : "Subject-dependent segment-level (matches paper)",
    "n_folds"        : N_FOLDS,
    "total_features" : int(features.shape[1]),
    "cbgg_features"  : int(feats_cbgg.shape[1]),
    "our_features"   : int(feats_ours.shape[1]),
    "cv_accuracy"    : f"{np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%",
    "cv_f1"          : f"{np.mean(f1s):.4f} ± {np.std(f1s):.4f}",
    "cv_auc"         : f"{np.mean(aucs):.4f} ± {np.std(aucs):.4f}",
    "paper_accuracy" : "98.10%",
    "paper_cv"       : "97.60%",
    "ablation"       : ablation_results,
}

with open(f"{RESULTS_DIR}/final_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("="*55)
print("  FINAL SUMMARY")
print("="*55)
print(f"  Model          : Enhanced CBGG (TensorFlow/Keras)")
print(f"  Dataset        : STEW (19200 segments, 45 subjects)")
print(f"  Split          : Subject-dependent segment-level")
print(f"  Total features : {features.shape[1]}")
print(f"    CBGG orig    : {feats_cbgg.shape[1]}")
print(f"    Our addition : {feats_ours.shape[1]}")
print(f"  CV Accuracy    : {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}%")
print(f"  Paper Accuracy : 98.10%")
print(f"  Paper CV       : 97.60%")
print(f"  CV F1 (wtd)    : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"  CV AUC (OvR)   : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"\n  Saved to       : {RESULTS_DIR}")
print("="*55)